# FIFA Player Scouting & Market Value Analysis
### Comprehensive Analysis and Machine Learning Pipeline

This notebook contains:
1. Data Cleaning and Feature Engineering
2. 10 Comprehensive Visualizations
3. Machine Learning (Random Forest) for Value Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

sns.set(style='darkgrid')
print('Libraries successfully imported.')

In [ ]:
# IMPORTANT: Update the filename below to your actual CSV filename
# df = pd.read_csv('your_file_name.csv')

# For demonstration, we will create dummy data based on your schema
data = {
    'Name': ['Player ' + str(i) for i in range(1000)],
    'Country': np.random.choice(['Egypt', 'Angola', 'Brazil', 'France', 'England'], 1000),
    'Position': np.random.choice(['LW', 'GK', 'CB', 'RB', 'CM', 'ST'], 1000),
    'Age': np.random.randint(16, 40, 1000),
    'Overall_Rating': np.random.randint(50, 95, 1000),
    'Team': np.random.choice(['Ittihad Alexandria', 'Al Ahly', 'Zamalek'], 1000),
    'Value Per M$': np.random.uniform(0.1, 100, 1000),
    'Total_Stats Score': np.random.randint(1000, 2500, 1000)
}
data['Future Potential'] = data['Overall_Rating'] + np.random.randint(0, 10, 1000)
df = pd.DataFrame(data)
df.head()

In [ ]:
# Creating useful metrics
df['Growth_Potential'] = df['Future Potential'] - df['Overall_Rating']
df['Value_Per_Rating_Point'] = df['Value Per M$'] / df['Overall_Rating']
print('Feature Engineering Complete.')

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(18, 30))

# 1. Rating Distribution
sns.histplot(df['Overall_Rating'], kde=True, ax=axes[0, 0], color='blue')
axes[0, 0].set_title('1. Overall Rating Distribution')

# 2. Potential vs Rating
sns.scatterplot(data=df, x='Overall_Rating', y='Future Potential', hue='Age', ax=axes[0, 1])
axes[0, 1].set_title('2. Current Rating vs. Future Potential')

# 3. Age vs Value
sns.lineplot(data=df, x='Age', y='Value Per M$', ax=axes[1, 0], color='green')
axes[1, 0].set_title('3. Market Value Trend by Age')

# 4. Position Breakdown
df['Position'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[1, 1])
axes[1, 1].set_title('4. Player Distribution by Position')

# 5. Top 5 Countries by Avg Rating
df.groupby('Country')['Overall_Rating'].mean().sort_values(ascending=False).head(5).plot(kind='bar', ax=axes[2, 0])
axes[2, 0].set_title('5. Top 5 Countries (Avg Rating)')

# 6. Correlation Heatmap
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', ax=axes[2, 1])
axes[2, 1].set_title('6. Attribute Correlation Matrix')

# 7. Boxplot Value by Position
sns.boxplot(data=df, x='Position', y='Value Per M$', ax=axes[3, 0])
axes[3, 0].set_title('7. Value Distribution by Position')

# 8. Total Stats vs Overall
sns.regplot(data=df, x='Total_Stats Score', y='Overall_Rating', ax=axes[3, 1], scatter_kws={'alpha':0.3})
axes[3, 1].set_title('8. Stats Score vs. Overall Rating')

# 9. Growth Gap
sns.violinplot(data=df, x='Position', y='Growth_Potential', ax=axes[4, 0])
axes[4, 0].set_title('9. Growth Potential Gap by Position')

# 10. Value Density
sns.kdeplot(data=df, x='Value Per M$', fill=True, ax=axes[4, 1])
axes[4, 1].set_title('10. Market Value Density')

plt.tight_layout()
plt.show()

## Machine Learning: Predicting Market Value

In [ ]:
# Encoding Position
le = LabelEncoder()
df['Position_Label'] = le.fit_transform(df['Position'])

# Select Features
X = df[['Age', 'Overall_Rating', 'Future Potential', 'Total_Stats Score', 'Position_Label']]
y = df['Value Per M$']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Training
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluation
predictions = model.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, predictions):.2f}')
print(f'R2 Score: {r2_score(y_test, predictions):.2f}')